# 01 · Extracción de los PML (MDA y MTR)

El reto pide pronosticar el PML horario del Mercado de Tiempo Real (MTR) para los 9 NodosP de la zona de carga Monclova, entre el 1 de enero de 2024 y el 30 de junio de 2025. Antes de modelar hay que asegurar dos cosas: que se usan exactamente los nodos que pide el reto y que no falta ninguna hora.

Los datos vienen del servicio web SW-PML de CENACE. El cliente (`src/pml/cenace_client.py`) pide bloques de 7 días, reintenta cuando el servidor falla y guarda la respuesta cruda en `data/raw/`, así que volver a correr este notebook no vuelve a llamar a la API. Además del periodo del reto se descargaron 14 días de diciembre de 2023: no se evalúan, solo sirven para calcular los rezagos de los primeros días de enero.

In [1]:
import sys
from datetime import date
from pathlib import Path
ROOT = Path.cwd().parent                            # carpeta Solution/
sys.path.insert(0, str(ROOT / 'src'))               # para importar el paquete del proyecto (src/pml)
import pandas as pd
from pml.cenace_client import descargar, NODOS, bloques

# Periodo del reto (2024-01-01 a 2025-06-30) más dos semanas antes, para que los primeros días tengan rezagos
INI, FIN = date(2023, 12, 18), date(2025, 6, 30)
print(len(list(bloques(INI, FIN))), 'bloques de 7 días por proceso (MDA y MTR)')

81 bloques de 7 días por proceso (MDA y MTR)


## 1. ¿Cuáles son los nodos de Monclova?

La lista de 9 nodos venía en el enunciado, pero no se da por buena sin compararla con el catálogo oficial de NodosP (versión 2025-06-18). Al filtrar la zona de carga MONCLOVA, se comprueba que coincide exactamente con la lista del reto.

In [2]:
# Catálogo oficial de NodosP de CENACE: ¿qué nodos pertenecen a la zona de carga Monclova?
CAT = ROOT / 'Catálogo NodosP Sistema Eléctrico Nacional (v2025-06-18).xlsx'
catalogo = pd.read_excel(CAT, sheet_name=0, header=1)
catalogo.columns = [str(c).strip() for c in catalogo.columns]
monclova = catalogo[catalogo['ZONA DE CARGA'].astype(str).str.upper().eq('MONCLOVA')]
cols = ['SISTEMA', 'CENTRO DE CONTROL REGIONAL', 'ZONA DE CARGA', 'CLAVE', 'NOMBRE',
        'NIVEL DE TENSIÓN (kV)', 'MUNICIPIO (INEGI)']
display(monclova[cols].reset_index(drop=True))

# Los nodos del catálogo deben ser exactamente los 9 del reto, y todos del Sistema Interconectado Nacional
nodos_catalogo = sorted(monclova['CLAVE'].str.strip())
assert nodos_catalogo == sorted(NODOS), (set(nodos_catalogo) ^ set(NODOS))
assert monclova['SISTEMA'].eq('SIN').all()
print(len(nodos_catalogo), 'nodos en el catálogo, iguales a la lista del reto; todos en el SIN')

,SISTEMA,CENTRO DE CONTROL REGIONAL,ZONA DE CARGA,CLAVE,NOMBRE,NIVEL DE TENSIÓN (kV),MUNICIPIO (INEGI)
0,SIN,NORESTE,MONCLOVA,06AEO-115,Acero,115.0,MONCLOVA
1,SIN,NORESTE,MONCLOVA,06AHM-400,Ahmsa,400.0,MONCLOVA
2,SIN,NORESTE,MONCLOVA,06FRO-115,Frontera,115.0,FRONTERA
3,SIN,NORESTE,MONCLOVA,06GRA-115,Granjas,115.0,MONCLOVA
4,SIN,NORESTE,MONCLOVA,06MON-115,Monclova,115.0,MONCLOVA
5,SIN,NORESTE,MONCLOVA,06MON-230,Monclova,230.0,MONCLOVA
6,SIN,NORESTE,MONCLOVA,06RGS-115,Regidores,115.0,FRONTERA
7,SIN,NORESTE,MONCLOVA,06TKS-115,Teksid,115.0,FRONTERA
8,SIN,NORESTE,MONCLOVA,06XOC-115,Xochitilli,115.0,MONCLOVA


9 nodos en el catálogo, iguales a la lista del reto; todos en el SIN


Coinciden los 9. Un detalle que importa más adelante: el catálogo no trae coordenadas, solo el municipio. Seis nodos están en Monclova y tres en Frontera, que forman la misma mancha urbana. Por eso, para el clima se toma una sola serie representativa de la zona y no una por subestación: no hay con qué distinguirlas.

## 2. Descarga

Si `src/pml/download_all.py` ya corrió, estas llamadas salen de la caché.

In [3]:
# Descarga (o lee de la caché en data/raw) los dos mercados y los guarda en formato largo
mtr = descargar('MTR', INI, FIN, ROOT / 'data/raw')
mda = descargar('MDA', INI, FIN, ROOT / 'data/raw')
for n, d in [('MTR', mtr), ('MDA', mda)]:
    d.to_parquet(ROOT / f'data/processed/pml_{n.lower()}.parquet', index=False)
    print(n, d.shape, d.fecha.min().date(), '→', d.fecha.max().date())

MTR (121176, 8) 2023-12-18 → 2025-06-30
MDA (121176, 8) 2023-12-18 → 2025-06-30


## 3. ¿Está completo?

Primero lo grueso: horas esperadas por nodo, duplicados y si el PML es igual a la suma de sus tres componentes (energía + pérdidas + congestión).

In [4]:
# Revisiones básicas: horas faltantes por nodo, filas duplicadas y que el PML sea la suma de sus tres componentes
dias = (FIN - INI).days + 1
esperado = dias * 24
for n, d in [('MTR', mtr), ('MDA', mda)]:
    c = d.groupby('nodo').size()
    print(n, '| horas esperadas por nodo:', esperado, '| faltantes:', (esperado - c).to_dict())
    print('   duplicados (nodo, fecha, hora):', d.duplicated(['nodo','fecha','hora']).sum(),
          '| máx |pml − (energía + pérdidas + congestión)|:', (d.pml - d[['pml_ene','pml_per','pml_cng']].sum(axis=1)).abs().max().round(3))

MTR | horas esperadas por nodo: 13464 | faltantes: {'06AEO-115': 0, '06AHM-400': 0, '06FRO-115': 0, '06GRA-115': 0, '06MON-115': 0, '06MON-230': 0, '06RGS-115': 0, '06TKS-115': 0, '06XOC-115': 0}
   duplicados (nodo, fecha, hora): 0 | máx |pml − (energía + pérdidas + congestión)|: 0.02
MDA | horas esperadas por nodo: 13464 | faltantes: {'06AEO-115': 0, '06AHM-400': 0, '06FRO-115': 0, '06GRA-115': 0, '06MON-115': 0, '06MON-230': 0, '06RGS-115': 0, '06TKS-115': 0, '06XOC-115': 0}
   duplicados (nodo, fecha, hora): 0 | máx |pml − (energía + pérdidas + congestión)|: 0.02


Luego lo fino, día por día y solo dentro del periodo del reto. Cada nodo debe tener cada día las horas 1 a 24, una sola vez, sin días faltantes ni fechas fuera de rango. Un total correcto podría esconder un día con 23 horas y otro con 25 (por ejemplo, por un cambio de horario mal resuelto); esta revisión lo descarta.

In [5]:
# Revisión estricta en el periodo del reto: cada nodo y cada día deben tener exactamente las horas 1 a 24, sin huecos
CH_INI, CH_FIN = pd.Timestamp('2024-01-01'), pd.Timestamp('2025-06-30')
dias_ch = pd.date_range(CH_INI, CH_FIN)
for n, d in [('MTR', mtr), ('MDA', mda)]:
    ch = d[(d.fecha >= CH_INI) & (d.fecha <= CH_FIN)]
    assert d.fecha.min() == pd.Timestamp(INI) and d.fecha.max() == pd.Timestamp(FIN)
    por_dia = ch.groupby(['nodo', 'fecha']).hora.agg(['count', 'min', 'max', 'nunique'])
    malos = por_dia[(por_dia['count'] != 24) | (por_dia['min'] != 1) | (por_dia['max'] != 24) | (por_dia['nunique'] != 24)]   # días incompletos
    dias_nodo = ch.groupby('nodo').fecha.nunique()
    print(n, '| filas:', len(ch), '(esperadas', len(dias_ch) * 24 * len(NODOS), ') | días-nodo incompletos:', len(malos),
          '| NaN:', int(ch[['pml','pml_ene','pml_per','pml_cng']].isna().sum().sum()))
    assert len(malos) == 0 and (dias_nodo == len(dias_ch)).all() and len(ch) == len(dias_ch) * 24 * len(NODOS)
print(f'Completo: {len(dias_ch)} días × 24 h × 9 nodos en MDA y en MTR')

MTR | filas: 118152 (esperadas 118152 ) | días-nodo incompletos: 0 | NaN: 0
MDA | filas: 118152 (esperadas 118152 ) | días-nodo incompletos: 0 | NaN: 0
Completo: 547 días × 24 h × 9 nodos en MDA y en MTR


**En resumen:** los datos están completos. Son 547 días × 24 horas × 9 nodos en los dos mercados, sin huecos, sin duplicados y con componentes que suman el PML. A partir de aquí todo el proyecto lee los parquet de `data/processed/`.